# User Clusters by Dominant Genre

In this analysis, we will identify user clusters based on their dominant anime genre preferences. We will extract genre data from a PostgreSQL database, clean it using Pandas, and then perform a federated query with DuckDB to join this data with user ratings stored in a Parquet file. Finally, we will visualize the distribution of users across different dominant genres using Seaborn and Matplotlib

In [ ]:
import pandas as pd
import duckdb
import os
import plotly.express as px
from lib import dbconnection

In [6]:
engine = dbconnection.create_db_engine()
df_details = pd.read_sql("SELECT mal_id, genres FROM details WHERE genres IS NOT NULL", engine)

df_details['genres'] = df_details['genres'].astype(str).str.replace(r"[\[\]'\" ]", "", regex=True)

path_ratings = os.path.normpath(os.path.join(os.getcwd(), "..", "..", "..", "cleaned_data", "ratings_cleaned.parquet")).replace('\\', '/')

## --- 2. Advanced Query (Cluster + Profiliation) ---


In addition to finding the dominant gender, we calculate behavior statistics for each user and aggregate them.

In [7]:
query_profiling = f"""
    WITH
    user_genres AS (
        SELECT
            r.username,
            unnest(string_split(d.genres, ',')) as genere
        FROM read_parquet('{path_ratings}') r
        JOIN df_details d ON r.anime_id = d.mal_id
        WHERE r.score >= 7
    ),
    user_top_genre AS (
        SELECT username, genere, COUNT(*) as conteggio
        FROM user_genres
        GROUP BY username, genere
    ),
    ranked_user_genres AS (
        SELECT username, genere,
            ROW_NUMBER() OVER(PARTITION BY username ORDER BY conteggio DESC) as rnk
        FROM user_top_genre
    ),
    user_activity AS (
        SELECT
            username,
            COUNT(*) as total_watched,
            AVG(score) as avg_score
        FROM read_parquet('{path_ratings}')
        WHERE score > 0
        GROUP BY username
    ),
    cluster_stats AS (
        SELECT
            r.genere as Cluster_Gusto,
            COUNT(r.username) as Numero_Utenti,
            AVG(u.total_watched) as Media_Anime_Visti,
            AVG(u.avg_score) as Voto_Medio_Dato
        FROM ranked_user_genres r
        JOIN user_activity u ON r.username = u.username
        WHERE r.rnk = 1
        GROUP BY r.genere
    )
    SELECT * FROM cluster_stats
    WHERE Numero_Utenti > 100
    ORDER BY Numero_Utenti DESC
"""

print("Esecuzione analisi profilazione cluster...")
df_analysis = duckdb.sql(query_profiling).df()
print(df_analysis.head())

Esecuzione analisi profilazione cluster...
  Cluster_Gusto  Numero_Utenti  Media_Anime_Visti  Voto_Medio_Dato
0        Action         180824         252.295779         7.715728
1        Comedy          38195         374.512711         7.666435
2         Drama          26222         169.354245         7.653951
3       Romance          14775         163.671404         8.057339
4       Fantasy          13109         164.235411         8.138476


## --- 3. Advanced Visualisation(Bubble Chart) ---

In [8]:
fig = px.scatter(
    df_analysis,
    x="Media_Anime_Visti",    # Modificato da Avg_Watched_animes
    y="Voto_Medio_Dato",      # Modificato da Avg_score_given
    size="Numero_Utenti",     # Modificato da Number_of_users
    color="Cluster_Gusto",
    hover_name="Cluster_Gusto",
    log_x=True,
    size_max=60,
    title="Community Profiling: Quality vs. Quantity",
    labels={
        "Media_Anime_Visti": "Avg Activity (Watched Anime - Log scale)",
        "Voto_Medio_Dato": "Avg Score Given",
        "Numero_Utenti": "Community Size",
        "Cluster_Gusto": "Favorite Genre"
    },
    template="plotly_white"
)

fig.update_layout(
    showlegend=True,
    height=700,
    legend_title_text='Genres (Click to show/hide)',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02
    )
)

fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br><br>" +
                  "Avg Activity: %{x:.1f} anime<br>" +
                  "Avg score: %{y:.2f}/10<br>" +
                  "Number of users: %{marker.size}<extra></extra>"
)

fig.write_html("../../graphs/cluster_profiling.html")
fig.show()